# Nested antiresonant nodeless fibers (NANF)

```{index} NANF
```
```{index} nested antiresonant fiber
```

A *nested antiresonant nodeless fiber* (NANF) improves on the plain antiresonant fiber of the [previous notebook](./3_1_arf.ipynb) by placing smaller capillaries *inside* each outer capillary. Each nested tube adds a further antiresonant reflection, so the leakage of the core mode drops dramatically, resulting in record-low losses in hollow-core fibers reported in [[Poletti 2014](#references)]. "Nodeless" refers to the tubes touching only the cladding, not each other, avoiding lossy glass nodes.

Because polarization effects and accurate loss prediction matter for these fibers, this notebook computes full Maxwell *vector* leaky modes (using `fibermode`'s  `leakyvecmodes`, whose search contour lives in the $Z^2$-plane: cf. the discussion in the [Bragg vector mode notebook](./2_2_bragg.ipynb)).

In [ ]:
import ngsolve as ng
import numpy as np
from ngsolve.webgui import Draw
from fibermode import NANF

## Constructing `NANF` objects

The geometry is specified by the core radius and a list of capillary specifications, one dict per nesting level: `N` tubes of radius `r`, wall thickness `t`, and embedding parameter `e`. The default is a six-tube outer ring with one nested tube each. Mesh sizes per region are also constructor arguments; the values below give a mesh coarse enough for a quick first run.

In [ ]:
A = NANF(glass_maxh=0.2, air_maxh=0.15, core_maxh=0.4)
Draw(A.mesh);

## Fundamental vector mode

The fundamental HE11-like mode is a *degenerate polarization pair*, so we search a small contour with a span of a few vectors. (The elliptic contour with `rhoinv` close to 1 is a thin ellipse hugging the real axis — an efficient choice here since the sought $Z^2$ values have tiny imaginary parts.)

```{index} leakyvecmodes; NANF
```

In [ ]:
betas, Zsqrs, Es, phis, R = A.leakyvecmodes(
    ctr=5.066,      # contour center in the Z^2 plane
    rad=0.005,
    alpha=5,
    p=2,
    nspan=3,
    npts=4,
    rhoinv=0.99,
    quadrule='ellipse_trapez_shift',
    niterations=10,
    nrestarts=0,
    stop_tol=1e-9,
    seed=1,
)

In [ ]:
print('CL [dB/m]:', 20 * np.array(betas).imag / np.log(10))

`leakyvecmodes` returns the transverse electric fields `Es` (in the H(curl) space) and the scaled longitudinal components `phis` (in H1). The transverse field of the fundamental pair:

In [ ]:
for e in Es:
    Draw(e.real, A.mesh, vectors={'grid_size': 200})

In [ ]:
for phi in phis:
    Draw(1e-1 * phi, A.mesh)

## Power flow

The Poynting vector of each computed mode is available through the `S` method; its longitudinal component $S_z$ shows where the mode's power actually flows — almost entirely within the air core:

In [ ]:
for e, phi, beta in zip(Es, phis, betas):
    Stv, Sz = A.S(e, phi, beta)
    Draw(1e-1 * Sz, A.mesh)

## Higher-order modes

Higher-order core modes live further along the real $Z^2$ axis and are substantially lossier — the basis of the fiber's effectively single-mode behavior. For this geometry a mode group sits near $Z^2 \approx 12.75$:

In [ ]:
betas_h, Zsqrs_h, Es_h, phis_h, _ = A.leakyvecmodes(
    ctr=12.75, rad=0.1, alpha=5, p=2,
    nspan=2, npts=2, rhoinv=0.99,
    quadrule='ellipse_trapez_shift',
    niterations=50, nrestarts=0, stop_tol=1e-9, seed=1)
print('CL [dB/m]:', 20 * np.array(betas_h).imag / np.log(10))

In [ ]:
for e in Es_h:
    Draw(e.imag, A.mesh, vectors={'grid_size': 1000})

Comparing the two CL printouts quantifies the higher-order-mode suppression. To study how loss varies with the nesting (e.g., removing the nested tubes, or adding a second nesting level), edit the `capillary_info` list in the constructor and rerun — each entry adds one antiresonant layer.

<a id='references'></a>
## References

- F. Poletti. *Nested antiresonant nodeless hollow core fiber.* Optics Express, 22:23807-23828, 2014.
- G. T. Jasion et al. *Hollow core NANF with 0.28 dB/km attenuation in the C and L bands.* Optical Fiber Communication Conference, 2020.